In [1]:
import numpy as np
from qutip import tensor, qeye, basis, sigmax, sigmay, sigmaz, destroy

n_qubits = 2
N_osc = 6   # truncation of harmonic oscillator

# single-qubit ops
sx = sigmax()
sy = sigmay()
sz = sigmaz()
id2 = qeye(2)

# oscillator ops
a  = destroy(N_osc)
adag = a.dag()
id_osc = qeye(N_osc)

def embed_qubit_op(single_op, i, n_qubits, osc_last=True):
    ops = []
    for k in range(n_qubits):
        ops.append(single_op if k == i else id2)
    qubits_op = tensor(ops)
    return tensor(qubits_op, id_osc) if osc_last else tensor(id_osc, qubits_op)


In [4]:
# Collective spin operators

from qutip import (qeye, tensor)
from qutip import propagator # or use (-1j * H * t).expm()

Sx = 0
Sz = 0
for i in range(n_qubits):
    Sx += embed_qubit_op(sx, i, n_qubits)
    Sz += embed_qubit_op(sz, i, n_qubits)

Hx = Sx           # global X field up to a scale factor
Hz = Sz           # global Z field

sp = (sx + 1j*sy)/2   # sigma_+
sm = (sx - 1j*sy)/2   # sigma_-

Jp = 0
for i in range(n_qubits):
    Jp += embed_qubit_op(sp, i, n_qubits)
Jm = Jp.dag()

g_TC = 1.0  # set coupling strength to 1; times will be “in units of 1/g_TC”
HTC = tensor(Jp, a) + tensor(Jm, adag)


def U_x(theta):
    # exp(-i theta Hx)
    return (-1j * Hx * theta).expm()

def U_z(theta):
    return (-1j * Hz * theta).expm()

def U_TC(tau):
    # tau is the total interaction time; since g_TC=1, this matches the dimensionless times in the paper up to 2π factors
    return (-1j * HTC * tau).expm()


In [5]:
def circuit_A(theta, theta_plus):
    # totally schematic; you need to match the exact order and angles from App. K
    U = qeye(2**n_qubits * N_osc)
    U = U_TC(tau1) * U_z(phi1) * U
    U = U_TC(tau2) * U_z(phi2) * U
    U = U_TC(tau3) * U_z(phi3) * U
    U = U_TC(tau4) * U_z(phi4) * U
    return U

# def circuit_F(...):
#     # similar pattern, based on its decomposition in the appendix
#     ...

def circuit_CZ():
    U = qeye(2**n_qubits * N_osc)
    U = U_z(theta1) * U
    U = circuit_A(theta, theta_plus) * U
    U = circuit_F(...) * U
    U = U_z(theta2) * U
    U = circuit_F(...).dag() * U
    return U


In [6]:
# basis state |0> of oscillator
vac = basis(N_osc, 0)

# construct logical 4x4 gate (on qubits) from the full unitary
def extract_qubit_unitary(U_full):
    # act on basis |00>, |01>, |10>, |11> tensored with |0_osc>
    qubit_dim = 2**n_qubits
    cols = []
    for b in range(qubit_dim):
        psi_qubits = basis(qubit_dim, b)
        psi_full_in = tensor(psi_qubits, vac)
        psi_full_out = U_full * psi_full_in
        # project onto oscillator vacuum
        amp_on_vac = psi_full_out.full().reshape(qubit_dim, N_osc)[:,0]
        cols.append(amp_on_vac)
    U_eff = np.column_stack(cols)  # 4x4 matrix
    return U_eff

In [7]:
from qiskit.circuit import QuantumCircuit, Gate, Parameter

n_qubits = 2
n_bus = 1     # oscillator line (just drawn as a qubit wire for visualization)

# Parameters
theta_x = Parameter('θx')
theta_z = Parameter('θz')
tau_tc  = Parameter('τ')

# --- Define symbolic/custom gates just for DRAWING ---

# Global X rotation gate (acts on all qubits)
Gx = Gate(name="Gx", num_qubits=n_qubits, params=[theta_x])

# Global Z rotation gate
Gz = Gate(name="Gz", num_qubits=n_qubits, params=[theta_z])

# Tavis–Cummings gate acting on all qubits + bus
TC = Gate(name="TC", num_qubits=n_qubits + n_bus, params=[tau_tc])

# --- Build a sample circuit ---

total_wires = n_qubits + n_bus
qc = QuantumCircuit(total_wires)

qubit_wires = list(range(n_qubits))           # e.g. [0,1]
bus_wire    = [n_qubits]                      # e.g. [2]

# Example sequence: Gz → TC → Gx → TC
qc.append(Gz, qubit_wires)
qc.barrier()
qc.append(TC, qubit_wires + bus_wire)
qc.barrier()
qc.append(Gx, qubit_wires)
qc.barrier()
qc.append(TC, qubit_wires + bus_wire)

print(qc)


     ┌─────────┐ ░ ┌────────┐ ░ ┌─────────┐ ░ ┌────────┐
q_0: ┤0        ├─░─┤0       ├─░─┤0        ├─░─┤0       ├
     │  Gz(θz) │ ░ │        │ ░ │  Gx(θx) │ ░ │        │
q_1: ┤1        ├─░─┤1 TC(τ) ├─░─┤1        ├─░─┤1 TC(τ) ├
     └─────────┘ ░ │        │ ░ └─────────┘ ░ │        │
q_2: ────────────░─┤2       ├─░─────────────░─┤2       ├
                 ░ └────────┘ ░             ░ └────────┘
